In [0]:
# dbutils.library.restartPython()

In [0]:
# =============================================================================
# FX Rate Forecasting with iTransformer
# =============================================================================
# This notebook trains iTransformer models to forecast GBP foreign exchange rates
# and UK interest rates across multiple prediction horizons (24, 48, 96 days).
# Model predictions are logged to MLflow and exported to the fx_predictions table
# for downstream analysis and evaluation.
# =============================================================================

import subprocess, os, mlflow, re
import numpy as np
import pandas as pd

# --- 1. Clone and prepare iTransformer repository ---
repo_path = "/tmp/iTransformer"

# Remove any existing clone to ensure clean state
subprocess.run(["rm", "-rf", repo_path])
clone_result = subprocess.run(["git", "clone", "https://github.com/thuml/iTransformer.git", repo_path],
                               capture_output=True, text=True)
assert os.path.exists(f"{repo_path}/run.py"), f"Clone failed: {clone_result.stderr[-500:]}"

# Fix deprecated numpy constant (np.Inf → np.inf)
tools_path = f"{repo_path}/utils/tools.py"
content = open(tools_path).read().replace("np.Inf", "np.inf")
open(tools_path, "w").write(content)
assert "np.Inf" not in open(tools_path).read()
print("Repo ready.")

# --- 2. Prepare training data from fx_gold table ---
# Load gold-standard FX rates with UK interest rates (bank_rate, sonia)
# Filter out null values and zeros to ensure clean training data
gold_df = spark.table("fx_gold").orderBy("rate_date").dropna()
level_cols = ["gbp_usd", "gbp_eur", "gbp_jpy", "gbp_cad", "gbp_chf",
              "gbp_aud", "gbp_sgd", "gbp_sek", "gbp_dkk", "bank_rate", "sonia"]
for col in level_cols:
    gold_df = gold_df.filter(gold_df[col] != 0)

# Export to CSV format expected by iTransformer (date column + features)
os.makedirs(f"{repo_path}/dataset/fx", exist_ok=True)
pdf = gold_df.toPandas().rename(columns={"rate_date": "date"})
pdf.to_csv(f"{repo_path}/dataset/fx/fx_gold.csv", index=False)
print("Exported. Row count:", len(pdf))

# --- 3. Train iTransformer models with multiple prediction horizons ---
# Key hyperparameters:
#   seq_len: Input sequence length (96 days of historical data)
#   pred_len: Forecast horizon (24, 48, or 96 days ahead)
#   label_len: Decoder start token length (half of pred_len)
#   features: 'M' = multivariate (all 20 variates predicted jointly)
#   inverse: Apply inverse transformation to denormalize predictions
seq_len = 96
for pred_len in [24, 48, 96]:
    label_len = pred_len // 2
    with mlflow.start_run(run_name=f"itransformer_seq{seq_len}_pred{pred_len}_inverse"):
        # Log hyperparameters to MLflow for experiment tracking
        mlflow.log_params({
            "seq_len": seq_len, "pred_len": pred_len, "label_len": label_len,
            "features": "M", "d_model": 128, "n_heads": 8, "inverse": True,
        })
        cmd = [
            "python", f"{repo_path}/run.py",
            "--is_training", "1",
            "--root_path", f"{repo_path}/dataset/fx/",
            "--data_path", "fx_gold.csv",
            "--model_id", f"fx_itransformer_pl{pred_len}_inv",
            "--model", "iTransformer",
            "--data", "custom",
            "--features", "M",
            "--target", "gbp_usd",
            "--freq", "d",
            "--seq_len", str(seq_len),
            "--label_len", str(label_len),
            "--pred_len", str(pred_len),
            "--enc_in", "20",
            "--dec_in", "20",
            "--c_out", "20",
            "--e_layers", "2",
            "--d_model", "128",
            "--n_heads", "8",
            "--des", "fx_run_inv",
            "--itr", "1",
            "--use_gpu", "False",
            "--inverse",
        ]
        # Run training via subprocess (iTransformer trains on CPU with use_gpu=False)
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=repo_path)
        print(f"\n=== pred_len={pred_len} (inverse) ===")
        print(result.stdout[-1500:])
        print("RETURN CODE:", result.returncode)
        if result.returncode != 0:
            print("---STDERR---")
            print(result.stderr[-2000:])
            continue
        
        # Extract and log test metrics (MSE, MAE) from training output
        mse_match = re.search(r"mse:([\d.]+)", result.stdout)
        mae_match = re.search(r"mae:([\d.]+)", result.stdout)
        if mse_match:
            mlflow.log_metric("test_mse", float(mse_match.group(1)))
        if mae_match:
            mlflow.log_metric("test_mae", float(mae_match.group(1)))
        # Log prediction artifacts (pred.npy, true.npy) to MLflow for each run
        results_dir = f"{repo_path}/results"
        if os.path.exists(results_dir):
            for setting_folder in os.listdir(results_dir):
                folder_path = os.path.join(results_dir, setting_folder)
                pred_path = os.path.join(folder_path, "pred.npy")
                true_path = os.path.join(folder_path, "true.npy")
                if os.path.exists(pred_path) and os.path.exists(true_path):
                    mlflow.log_artifact(pred_path, artifact_path="forecast")
                    mlflow.log_artifact(true_path, artifact_path="forecast")
            print("Forecast artifacts logged.")
        else:
            print("WARNING: no results/ folder — cwd fix may not have applied.")

# --- 4. Export predictions to fx_predictions table ---
# Parse prediction arrays from all trained models and reshape into row format
# for downstream analysis. Each row represents one variate's prediction at one
# forecast horizon (day_ahead) for one model configuration (pred_len_config).

# Variate names match the column order in iTransformer's multivariate output
VARIATE_NAMES = [
    "gbp_eur", "gbp_jpy", "gbp_cad", "gbp_chf", "gbp_aud",
    "gbp_sgd", "gbp_sek", "gbp_dkk", "bank_rate", "sonia",
    "gbp_usd_logret", "gbp_eur_logret", "gbp_jpy_logret", "gbp_cad_logret",
    "gbp_chf_logret", "gbp_aud_logret", "gbp_sgd_logret", "gbp_sek_logret",
    "gbp_dkk_logret", "gbp_usd",
]

results_dir = f"{repo_path}/results"
setting_folders = [f for f in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, f))]
assert setting_folders, "No results/ folder right after training — investigate before going further."

# Reshape prediction arrays into tabular format
# pred.npy shape: (num_windows, pred_len, num_variates)
# We use the last window (most recent forecast) from each model
rows = []
for folder in setting_folders:
    # Extract prediction length from folder name (e.g., fx_itransformer_pl96_inv)
    pred_len_match = re.search(r"_pl(\d+)", folder)
    pred_len_val = pred_len_match.group(1) if pred_len_match else "unknown"
    
    # Load predictions and ground truth from numpy arrays
    pred = np.load(os.path.join(results_dir, folder, "pred.npy"))
    true = np.load(os.path.join(results_dir, folder, "true.npy"))
    
    # Use the last forecast window (most recent)
    last_window = -1
    for variate_idx, variate_name in enumerate(VARIATE_NAMES):
        pred_series = pred[last_window, :, variate_idx]
        true_series = true[last_window, :, variate_idx]
        # Create one row per forecast step (day 1, day 2, ..., day pred_len)
        for step, (p, t) in enumerate(zip(pred_series, true_series)):
            rows.append({
                "pred_len_config": pred_len_val, "variate": variate_name,
                "day_ahead": step + 1, "actual": float(t), "forecast": float(p),
            })

# Write to Delta table (overwrites existing data)
predictions_df = pd.DataFrame(rows)
spark.createDataFrame(predictions_df).write.format("delta").mode("overwrite").saveAsTable("fx_predictions")
print("Wrote fx_predictions:", len(predictions_df), "rows")